# Turning RGB Baseline Comparison – Grouped 5-Fold Cross-Validation

This notebook evaluates the classical anomaly-detection baselines
on the public turning dataset using the full three-channel RGB
representation.

The outer cross-validation folds are grouped by machining run and
are identical to those used for the CNN autoencoder evaluation.
Within each development fold, separate source-run groups are used
for one-class model fitting and decision-threshold calibration.

PCA, one-class SVM, and isolation forest are fitted exclusively on
nominal samples from the respective training groups.

In [1]:
from pathlib import Path
import json

import numpy as np
import pandas as pd
from PIL import Image
from sklearn.decomposition import PCA
from sklearn.ensemble import IsolationForest
from sklearn.metrics import average_precision_score, confusion_matrix, f1_score, precision_score, recall_score, precision_recall_curve
from sklearn.model_selection import GroupShuffleSplit
from sklearn.preprocessing import StandardScaler
from sklearn.svm import OneClassSVM

In [2]:
CWD = Path.cwd().resolve()
REPO_ROOT = CWD if (CWD / 'data').exists() else CWD.parent

BASELINE_DIR = REPO_ROOT / 'reports' / 'baselines'
THRESHOLD_DIR = REPO_ROOT / 'reports' / 'thresholds'
TABLE_DIR = REPO_ROOT / 'reports' / 'tables'

IMAGE_SIZE = (150, 100)
PCA_COMPONENTS = 32
SEED = 42

N_OUTER_FOLDS = 5
N_INNER_FOLDS = 5

TURNING_MANIFEST = (
    REPO_ROOT
    / 'reports'
    / 'manifests'
    / 'turning_split_seed42.csv'
)

BROACH_MANIFEST = (
    REPO_ROOT
    / 'reports'
    / 'manifests'
    / 'broach_dataset_split_seed42.csv'
)

DATASETS = [
    {
        'name': 'turning',
        'manifest_path': TURNING_MANIFEST,
        'source_dataset': 'turning',
    },
    {
        'name': 'broach_dataset',
        'manifest_path': BROACH_MANIFEST,
        'source_dataset': 'broach_dataset',
    },
]

BASELINE_DIR.mkdir(parents=True, exist_ok=True)
THRESHOLD_DIR.mkdir(parents=True, exist_ok=True)
TABLE_DIR.mkdir(parents=True, exist_ok=True) 

In [3]:
# set constants for CV
INNER_CALIBRATION_SIZE = 0.25
INNER_SEED = 42

GROUP_COLUMN = "source_run"

#Load Turning CV Folds
TURNING_CV_ASSIGNMENTS = (
    REPO_ROOT
    / "reports"
    / "cv"
    / "turning_ae_grouped_cv_assignments.csv"
)

turning_manifest = pd.read_csv(
    TURNING_CV_ASSIGNMENTS
)
turning_manifest["dataset"] = "turning"

required_columns = {
    "sample_id",
    "image_path",
    "label",
    "target",
    "source_run",
    "cv_fold",
}

missing = required_columns - set(
    turning_manifest.columns
)

if missing:
    raise ValueError(
        f"Missing columns: {missing}"
    )

In [ ]:
def load_turning_manifest(config: dict) -> pd.DataFrame:
    manifest = pd.read_csv(config["manifest_path"])

    manifest = manifest[
        manifest["source_dataset"] == config["source_dataset"]
    ].copy()

    manifest["dataset"] = manifest["source_dataset"]
    manifest["target"] = (manifest["label"] == "chatter").astype(int)

    return manifest[
        [
            "dataset",
            "sample_id",
            "split",
            "label",
            "target",
            "image_path",
        ]
    ]

def load_broach_manifest(config: dict) -> pd.DataFrame:
    manifest = pd.read_csv(config["manifest_path"])

    manifest = manifest.copy()
    manifest["dataset"] = "broach_dataset"
    manifest["sample_id"] = (
        manifest["filename"]
        .str.replace(".png", "", regex=False)
    )

    manifest["target"] = manifest["label"]

    manifest["label"] = manifest["target"].map({
        0: "normal",
        1: "anomaly"
    })

    manifest["image_path"] = manifest.apply(
        lambda row:
            f"data/03_broach_dataset/train/{row['filename']}"
            if row["split"] == "train"
            else f"data/03_broach_dataset/test/{row['filename']}",
        axis=1,
    )

    return manifest[
        [
            "dataset",
            "sample_id",
            "split",
            "label",
            "target",
            "image_path",
            "source_run",
        ]
    ]


def build_dataset_manifest(config: dict) -> pd.DataFrame:

    if config["name"] == "turning":
        return load_turning_manifest(config)

    if config["name"] == "broach_dataset":
        return load_broach_manifest(config)

    raise ValueError(f"Unknown dataset: {config['name']}")


In [17]:
def open_resized_image(image_path: Path, mode: str) -> np.ndarray:
    image = Image.open(image_path).convert(mode).resize(IMAGE_SIZE)
    return np.asarray(image, dtype='float32') / 255.0


def image_vector(image_path: Path) -> np.ndarray:
    return open_resized_image(image_path, 'RGB').reshape(-1)


def channel_descriptor(arr: np.ndarray) -> np.ndarray:
    grad_y, grad_x = np.gradient(arr)

    hist, _ = np.histogram(
        arr,
        bins=32,
        range=(0.0, 1.0),
        density=True,
    )

    percentiles = np.percentile(
        arr,
        [1, 5, 25, 50, 75, 95, 99],
    )

    summary = np.array([
        arr.mean(),
        arr.std(),
        arr.min(),
        arr.max(),
        np.abs(grad_x).mean(),
        np.abs(grad_y).mean(),
        np.sqrt(grad_x ** 2 + grad_y ** 2).mean(),
    ], dtype='float32')

    return np.concatenate([
        summary,
        percentiles.astype('float32'),
        hist.astype('float32'),
        arr.mean(axis=0),
        arr.mean(axis=1),
        arr.std(axis=0),
        arr.std(axis=1),
    ]).astype('float32')

def image_descriptor(image_path: Path) -> np.ndarray:
    arr = open_resized_image(image_path, 'RGB')

    channel_features = [
        channel_descriptor(arr[:, :, channel])
        for channel in range(3)
    ]

    return np.concatenate(channel_features).astype('float32')


def load_matrix(rows: pd.DataFrame, extractor) -> np.ndarray:
    values = [extractor(REPO_ROOT / image_path) for image_path in rows['image_path']]
    if not values:
        raise ValueError('No images loaded.')
    return np.stack(values, axis=0)

In [18]:
def select_best_f1_threshold(y_true: np.ndarray, scores: np.ndarray) -> dict:
    precision, recall, thresholds = precision_recall_curve(y_true, scores)
    if len(thresholds) == 0:
        threshold = float(np.max(scores))
        return {
            'threshold': threshold,
            'validation_f1': 0.0,
            'validation_precision': 0.0,
            'validation_recall': 0.0,
        }

    f1 = 2 * precision[:-1] * recall[:-1] / np.maximum(precision[:-1] + recall[:-1], 1e-12)
    best_idx = int(np.nanargmax(f1))
    return {
        'threshold': float(thresholds[best_idx]),
        'validation_f1': float(f1[best_idx]),
        'validation_precision': float(precision[best_idx]),
        'validation_recall': float(recall[best_idx]),
    }


def evaluate_at_threshold(y_true: np.ndarray, scores: np.ndarray, threshold: float) -> dict:
    y_pred = (scores >= threshold).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[0, 1]).ravel()
    return {
        'pr_auc': float(average_precision_score(y_true, scores)),
        'f1': float(f1_score(y_true, y_pred, zero_division=0)),
        'precision': float(precision_score(y_true, y_pred, zero_division=0)),
        'recall': float(recall_score(y_true, y_pred, zero_division=0)),
        'tn': int(tn),
        'fp': int(fp),
        'fn': int(fn),
        'tp': int(tp),
    }


def append_score_frame(
    score_frames: list[pd.DataFrame],
    rows: pd.DataFrame,
    method: str,
    score_values: np.ndarray,
    outer_fold: int | None = None,
) -> None:

    score_frames.append(pd.DataFrame({
        'dataset': rows['dataset'].to_numpy(),
        'method': method,
        'sample_id': rows['sample_id'].to_numpy(),
        'split': rows['split'].to_numpy(),
        'label': rows['label'].to_numpy(),
        'target': rows['target'].to_numpy(),
        'score_value': score_values,
        'outer_fold': outer_fold,
    }))

In [19]:
# fit descriptor models
def fit_descriptor_baselines(
    train_nominal: pd.DataFrame,
):
    X_train = load_matrix(
        train_nominal,
        image_descriptor,
    )

    scaler = StandardScaler().fit(
        X_train
    )

    X_train_scaled = scaler.transform(
        X_train
    )

    ocsvm = OneClassSVM(
        kernel="rbf",
        gamma="scale",
        nu=0.05,
    ).fit(
        X_train_scaled
    )

    iforest = IsolationForest(
        random_state=SEED,
        contamination="auto",
    ).fit(
        X_train_scaled
    )

    return scaler, ocsvm, iforest 

In [20]:
# use descriptor models
def score_descriptor_baselines(
    scaler,
    ocsvm,
    iforest,
    evaluation_fold: pd.DataFrame,
):
    X_eval = load_matrix(
        evaluation_fold,
        image_descriptor,
    )

    X_eval = scaler.transform(
        X_eval
    )

    return {
        "one_class_svm_rgb_features":
            -ocsvm.decision_function(X_eval),

        "isolation_forest_rgb_features":
            -iforest.decision_function(X_eval),
    }

In [21]:
# fit pca descriptor
def fit_pca_baseline(
    train_nominal: pd.DataFrame,
):
    X_train = load_matrix(
        train_nominal,
        image_vector,
    )

    n_components = min(
        PCA_COMPONENTS,
        X_train.shape[0],
        X_train.shape[1],
    )

    pca = PCA(
        n_components=n_components,
        random_state=SEED,
        svd_solver="randomized",
    ).fit(
        X_train
    )

    return pca

In [22]:
# fit pca descriptor
def fit_pca_baseline(
    train_nominal: pd.DataFrame,
):
    X_train = load_matrix(
        train_nominal,
        image_vector,
    )

    n_components = min(
        PCA_COMPONENTS,
        X_train.shape[0],
        X_train.shape[1],
    )

    pca = PCA(
        n_components=n_components,
        random_state=SEED,
        svd_solver="randomized",
    ).fit(
        X_train
    )

    return pca

In [23]:
# use pca
def score_pca_baseline(
    pca,
    evaluation_fold: pd.DataFrame,
):
    X_eval = load_matrix(
        evaluation_fold,
        image_vector,
    )

    recon = pca.inverse_transform(
        pca.transform(X_eval)
    )

    return np.mean(
        (X_eval - recon) ** 2,
        axis=1,
    )

In [24]:
def run_turning_baselines_grouped_cv(
    manifest: pd.DataFrame,
) -> tuple[pd.DataFrame, pd.DataFrame, dict]:

    dataset_name = "turning"

    metric_rows = []
    score_frames = []
    thresholds = {}

    # ========================================================
    # Outer grouped folds
    # ========================================================

    for outer_fold in range(
        1,
        N_OUTER_FOLDS + 1,
    ):

        print("\n" + "=" * 70)
        print(
            f"TURNING BASELINES - "
            f"OUTER FOLD {outer_fold}/{N_OUTER_FOLDS}"
        )
        print("=" * 70)

        # ----------------------------------------------------
        # Same outer folds as CNN-AE
        # ----------------------------------------------------

        outer_dev = (
            manifest[
                manifest["cv_fold"] != outer_fold
            ]
            .copy()
            .reset_index(drop=True)
        )

        outer_test = (
            manifest[
                manifest["cv_fold"] == outer_fold
            ]
            .copy()
            .reset_index(drop=True)
        )

        # ----------------------------------------------------
        # Same group-wise calibration split as CNN-AE
        # ----------------------------------------------------

        inner_splitter = GroupShuffleSplit(
            n_splits=1,
            test_size=INNER_CALIBRATION_SIZE,
            random_state=INNER_SEED + outer_fold,
        )

        inner_train_idx, calibration_idx = next(
            inner_splitter.split(
                outer_dev,
                y=outer_dev["target"],
                groups=outer_dev[GROUP_COLUMN],
            )
        )

        inner_train = (
            outer_dev
            .iloc[inner_train_idx]
            .copy()
            .reset_index(drop=True)
        )

        calibration = (
            outer_dev
            .iloc[calibration_idx]
            .copy()
            .reset_index(drop=True)
        )

        # ----------------------------------------------------
        # One-class training: nominal only
        # ----------------------------------------------------

        train_nominal = (
            inner_train[
                inner_train["target"] == 0
            ]
            .copy()
            .reset_index(drop=True)
        )

        # ----------------------------------------------------
        # Leakage checks
        # ----------------------------------------------------

        train_groups = set(
            inner_train[GROUP_COLUMN].unique()
        )

        calibration_groups = set(
            calibration[GROUP_COLUMN].unique()
        )

        test_groups = set(
            outer_test[GROUP_COLUMN].unique()
        )

        assert train_groups.isdisjoint(
            calibration_groups
        )

        assert train_groups.isdisjoint(
            test_groups
        )

        assert calibration_groups.isdisjoint(
            test_groups
        )

        assert (
            train_nominal["target"] == 0
        ).all()

        assert set(
            calibration["target"].unique()
        ) == {0, 1}

        assert set(
            outer_test["target"].unique()
        ) == {0, 1}

        # ====================================================
        # FIT BASELINES ONCE
        # ====================================================

        scaler, ocsvm, iforest = (
            fit_descriptor_baselines(
                train_nominal
            )
        )

        pca = fit_pca_baseline(
            train_nominal
        )

        # ====================================================
        # CALIBRATION SCORES
        # ====================================================

        y_cal = calibration[
            "target"
        ].to_numpy()

        descriptor_cal = (
            score_descriptor_baselines(
                scaler,
                ocsvm,
                iforest,
                calibration,
            )
        )

        pca_cal = score_pca_baseline(
            pca,
            calibration,
        )

        calibration_scores = {
            **descriptor_cal,
            "pca_rgb_reconstruction": pca_cal,
        }

        # ----------------------------------------------------
        # Threshold calibration
        # ----------------------------------------------------

        final_thresholds = {}

        for method, scores in (
            calibration_scores.items()
        ):

            selected = (
                select_best_f1_threshold(
                    y_cal,
                    scores,
                )
            )

            final_thresholds[
                method
            ] = float(
                selected["threshold"]
            )

        thresholds[
            f"outer_fold_{outer_fold}"
        ] = final_thresholds

        # ====================================================
        # OUTER TEST SCORES
        # ====================================================

        y_test = (
            outer_test["target"]
            .to_numpy()
        )

        # Descriptor-based baselines:
        # use the SAME fitted scaler/models as for calibration
        descriptor_test = (
            score_descriptor_baselines(
                scaler,
                ocsvm,
                iforest,
                outer_test,
            )
        )

        # PCA:
        # use the SAME fitted PCA model as for calibration
        pca_test = (
            score_pca_baseline(
                pca,
                outer_test,
            )
        )

        test_scores = {
            **descriptor_test,
            "pca_rgb_reconstruction": pca_test,
        }

        # ====================================================
        # FINAL OUTER-FOLD EVALUATION
        # ====================================================

        for method, scores in (
            test_scores.items()
        ):

            threshold = (
                final_thresholds[
                    method
                ]
            )

            metrics = (
                evaluate_at_threshold(
                    y_test,
                    scores,
                    threshold,
                )
            )

            score_name = (
                "reconstruction_mse"
                if method
                == "pca_rgb_reconstruction"
                else "anomaly_score"
            )

            metric_rows.append({
                "dataset": dataset_name,
                "outer_fold": outer_fold,
                "method": method,
                "score": score_name,
                "threshold": threshold,
                **metrics,
            })

            append_score_frame(
                score_frames,
                outer_test,
                method,
                scores,
                outer_fold,
            )

    metrics = pd.DataFrame(
        metric_rows
    )

    scores = pd.concat(
        score_frames,
        ignore_index=True,
    )

    return metrics, scores, thresholds

In [25]:
turning_metrics, turning_scores, turning_thresholds = (
    run_turning_baselines_grouped_cv(
        turning_manifest
    )
)

display(turning_metrics)


TURNING BASELINES - OUTER FOLD 1/5

TURNING BASELINES - OUTER FOLD 2/5

TURNING BASELINES - OUTER FOLD 3/5

TURNING BASELINES - OUTER FOLD 4/5

TURNING BASELINES - OUTER FOLD 5/5


,dataset,outer_fold,method,score,threshold,pr_auc,f1,precision,recall,tn,fp,fn,tp
0,turning,1,one_class_svm_rgb_features,anomaly_score,0.674353,0.983824,0.866667,0.866667,0.866667,119,2,2,13
1,turning,1,isolation_forest_rgb_features,anomaly_score,0.097137,0.991389,0.928571,1.000000,0.866667,121,0,2,13
2,turning,1,pca_rgb_reconstruction,reconstruction_mse,0.001706,0.909830,0.882353,0.789474,1.000000,117,4,0,15
3,turning,2,one_class_svm_rgb_features,anomaly_score,0.881064,0.696230,0.428571,1.000000,0.272727,124,0,8,3
4,turning,2,isolation_forest_rgb_features,anomaly_score,0.224717,0.689366,0.428571,1.000000,0.272727,124,0,8,3
5,turning,2,pca_rgb_reconstruction,reconstruction_mse,0.004812,0.618672,0.428571,1.000000,0.272727,124,0,8,3
6,turning,3,one_class_svm_rgb_features,anomaly_score,0.171221,0.640554,0.588235,0.416667,1.000000,104,14,0,10
7,turning,3,isolation_forest_rgb_features,anomaly_score,0.122424,0.942136,0.842105,0.888889,0.800000,117,1,2,8
8,turning,3,pca_rgb_reconstruction,reconstruction_mse,0.002085,0.519774,0.608696,0.538462,0.700000,112,6,3,7
9,turning,4,one_class_svm_rgb_features,anomaly_score,0.286741,0.878788,0.758621,0.611111,1.000000,102,7,0,11


In [26]:
METRICS_PATH = (
    TABLE_DIR
    / "metrics_turning_grouped_cv_rgb.csv"
)

SCORES_PATH = (
    BASELINE_DIR
    / "baseline_scores_turning_grouped_cv_rgb.csv"
)

THRESHOLDS_PATH = (
    THRESHOLD_DIR
    / "baseline_thresholds_turning_grouped_cv_rgb.json"
)

turning_metrics.to_csv(
    METRICS_PATH,
    index=False,
)

turning_scores.to_csv(
    SCORES_PATH,
    index=False,
)

with THRESHOLDS_PATH.open("w") as f:
    json.dump(
        turning_thresholds,
        f,
        indent=2,
    )

print(f"Wrote {METRICS_PATH}")
print(f"Wrote {SCORES_PATH}")
print(f"Wrote {THRESHOLDS_PATH}")

turning_summary = (
    turning_metrics
    .groupby("method")
    .agg(
        pr_auc_mean=("pr_auc", "mean"),
        pr_auc_std=("pr_auc", "std"),
        f1_mean=("f1", "mean"),
        f1_std=("f1", "std"),
        precision_mean=("precision", "mean"),
        precision_std=("precision", "std"),
        recall_mean=("recall", "mean"),
        recall_std=("recall", "std"),
    )
    .reset_index()
)

display(turning_summary)

Wrote C:\Users\Zeleny\Documents\01_VSCode\spectrogram-anomaly-ae\reports\tables\metrics_turning_grouped_cv_rgb.csv
Wrote C:\Users\Zeleny\Documents\01_VSCode\spectrogram-anomaly-ae\reports\baselines\baseline_scores_turning_grouped_cv_rgb.csv
Wrote C:\Users\Zeleny\Documents\01_VSCode\spectrogram-anomaly-ae\reports\thresholds\baseline_thresholds_turning_grouped_cv_rgb.json


,method,pr_auc_mean,pr_auc_std,f1_mean,f1_std,precision_mean,precision_std,recall_mean,recall_std
0,isolation_forest_rgb_features,0.881970,0.137520,0.746813,0.192833,0.878914,0.132052,0.716450,0.279398
1,one_class_svm_rgb_features,0.753695,0.172332,0.646937,0.169438,0.701966,0.230812,0.742165,0.315429
2,pca_rgb_reconstruction,0.723586,0.192068,0.613999,0.165428,0.661354,0.233866,0.708831,0.307694
